In [ ]:
from pyspark.sql.functions import col, count, countDistinct, min, max, avg, when, isnan
from pyspark.sql.types import DateType, DecimalType, IntegerType, BooleanType

CATALOG = "czech_fintech"
SILVER  = "silver"

def assert_check(condition, msg):
    if not condition:
        raise AssertionError(f"❌ FAIL — {msg}")
    print(f"✅ {msg}")

print("=" * 80)
print("SILVER LAYER QUALITY CHECKS")
print("=" * 80)

In [ ]:
# ============================================================================
# 1. TRANSACTIONS
# ============================================================================
print("\n1️⃣  TRANSACTIONS")
trans = spark.table(f"{CATALOG}.{SILVER}.transactions")
trans_count = trans.count()

# row count
assert_check(trans_count > 0, f"transactions non vuota ({trans_count} rows)")

# unicità
distinct_ids = trans.select("transaction_id").distinct().count()
assert_check(distinct_ids == trans_count, f"transaction_id unico ({distinct_ids}/{trans_count})")

# tipi colonne chiave
schema = {f.name: type(f.dataType) for f in trans.schema.fields}
assert_check(schema["date"]    == DateType,    "date è DateType")
assert_check(schema["amount"]  == DecimalType, "amount è DecimalType")
assert_check(schema["balance"] == DecimalType, "balance è DecimalType")

# client_id popolato (join via disp)
nulls_client = trans.filter(col("client_id").isNull()).count()
pct_covered  = round((trans_count - nulls_client) / trans_count * 100, 1)
assert_check(pct_covered >= 95, f"client_id copertura ≥ 95% (attuale: {pct_covered}%)")

# date range sensato
date_min, date_max = trans.select(min("date"), max("date")).first()
assert_check(str(date_min)[:4] == "1993", f"date_min anno atteso 1993 (trovato {date_min})")
assert_check(str(date_max)[:4] <= "1999", f"date_max anno ≤ 1999 (trovato {date_max})")

# partizioni year/month presenti
assert_check("year"  in trans.columns, "colonna year presente")
assert_check("month" in trans.columns, "colonna month presente")

trans.select(min("date").alias("date_min"), max("date").alias("date_max"),
             min("amount").alias("amt_min"), max("amount").alias("amt_max")).show()

In [ ]:
# ============================================================================
# 2. ACCOUNT — SCD Type 2
# ============================================================================
print("\n2️⃣  ACCOUNT")
account = spark.table(f"{CATALOG}.{SILVER}.account")

assert_check(account.count() > 0, f"account non vuota ({account.count()} rows)")

# colonne SCD presenti
for c in ["valid_from", "valid_to", "is_current"]:
    assert_check(c in account.columns, f"colonna SCD '{c}' presente")

schema_acc = {f.name: type(f.dataType) for f in account.schema.fields}
assert_check(schema_acc["date"]       == DateType,    "date è DateType")
assert_check(schema_acc["is_current"] == BooleanType, "is_current è BooleanType")

# ogni account_id ha esattamente una riga is_current=true
dupes = (account.filter(col("is_current") == True)
         .groupBy("account_id").count()
         .filter(col("count") > 1).count())
assert_check(dupes == 0, "nessun account_id con più di una riga is_current=true")

# no NULL su chiavi
assert_check(account.filter(col("account_id").isNull()).count() == 0, "account_id senza NULL")

account.groupBy("frequency").count().orderBy("frequency").show()

In [ ]:
# ============================================================================
# 3. CLIENT — gender + birth_date
# ============================================================================
print("\n3️⃣  CLIENT")
client = spark.table(f"{CATALOG}.{SILVER}.client")

assert_check(client.count() > 0, f"client non vuota ({client.count()} rows)")

schema_cli = {f.name: type(f.dataType) for f in client.schema.fields}
assert_check(schema_cli["birth_date"] == DateType, "birth_date è DateType")

# gender solo M/F
invalid_gender = client.filter(~col("gender").isin("M", "F")).count()
assert_check(invalid_gender == 0, "gender contiene solo M/F")

# split M/F ragionevole
gender_dist = {r["gender"]: r["count"] for r in client.groupBy("gender").count().collect()}
assert_check(gender_dist.get("M", 0) > 0, f"clienti M presenti ({gender_dist.get('M', 0)})")
assert_check(gender_dist.get("F", 0) > 0, f"clienti F presenti ({gender_dist.get('F', 0)})")

# birth_date range sensato (dataset anni '90, clienti nati ~1910-1980)
bd_min, bd_max = client.select(min("birth_date"), max("birth_date")).first()
assert_check(str(bd_min)[:4] >= "1910", f"birth_date_min ≥ 1910 (trovato {bd_min})")
assert_check(str(bd_max)[:4] <= "1980", f"birth_date_max ≤ 1980 (trovato {bd_max})")

client.groupBy("gender").count().show()

In [ ]:
# ============================================================================
# 4. LOAN
# ============================================================================
print("\n4️⃣  LOAN")
loan = spark.table(f"{CATALOG}.{SILVER}.loan")

assert_check(loan.count() > 0, f"loan non vuota ({loan.count()} rows)")

schema_loan = {f.name: type(f.dataType) for f in loan.schema.fields}
assert_check(schema_loan["date"]     == DateType,    "date è DateType")
assert_check(schema_loan["amount"]   == DecimalType, "amount è DecimalType")
assert_check(schema_loan["duration"] == IntegerType, "duration è IntegerType")
assert_check(schema_loan["payments"] == DecimalType, "payments è DecimalType")

# status solo A/B/C/D
invalid_status = loan.filter(~col("status").isin("A", "B", "C", "D")).count()
assert_check(invalid_status == 0, "loan status contiene solo A/B/C/D")

loan.groupBy("status").count().orderBy("status").show()

# ============================================================================
# 5. ORDER
# ============================================================================
print("\n5️⃣  ORDER")
order = spark.table(f"{CATALOG}.{SILVER}.order")

assert_check(order.count() > 0, f"order non vuota ({order.count()} rows)")
schema_ord = {f.name: type(f.dataType) for f in order.schema.fields}
assert_check(schema_ord["amount"] == DecimalType, "amount è DecimalType")
assert_check(order.filter(col("amount").isNull()).count() == 0, "amount senza NULL")

# ============================================================================
# 6. CARD
# ============================================================================
print("\n6️⃣  CARD")
card = spark.table(f"{CATALOG}.{SILVER}.card")

assert_check(card.count() > 0, f"card non vuota ({card.count()} rows)")
schema_card = {f.name: type(f.dataType) for f in card.schema.fields}
assert_check(schema_card["issued"] == DateType, "issued è DateType")

card.groupBy("type").count().show()

# ============================================================================
# 7. DISP
# ============================================================================
print("\n7️⃣  DISP")
disp = spark.table(f"{CATALOG}.{SILVER}.disp")

assert_check(disp.count() > 0, f"disp non vuota ({disp.count()} rows)")
invalid_type = disp.filter(~col("type").isin("OWNER", "DISPONENT")).count()
assert_check(invalid_type == 0, "disp type contiene solo OWNER/DISPONENT")

# ============================================================================
# 8. DISTRICT
# ============================================================================
print("\n8️⃣  DISTRICT")
district = spark.table(f"{CATALOG}.{SILVER}.district")

assert_check(district.count() > 0, f"district non vuota ({district.count()} rows)")
assert_check(district.filter(col("A1").isNull()).count() == 0, "A1 (district_id) senza NULL")

district.select("A1", "A2", "A3").limit(5).show()

In [ ]:
# ============================================================================
# 9. JOIN COVERAGE — transactions → account → client → district
# ============================================================================
print("\n9️⃣  JOIN COVERAGE")

trans_total = trans.count()

# transactions → account
matched_account = trans.join(account.filter(col("is_current")).select("account_id"), "account_id", "inner").count()
pct_account = round(matched_account / trans_total * 100, 1)
assert_check(pct_account == 100.0, f"100% transactions matchano un account ({pct_account}%)")

# transactions → client (via client_id già joinato in silver)
matched_client = trans.filter(col("client_id").isNotNull()).count()
pct_client = round(matched_client / trans_total * 100, 1)
assert_check(pct_client >= 95, f"client_id copertura ≥ 95% ({pct_client}%)")

# account → district
account_total = account.filter(col("is_current")).count()
matched_district = (account.filter(col("is_current"))
                    .join(district.select(col("A1").alias("district_id")), "district_id", "inner")
                    .count())
pct_district = round(matched_district / account_total * 100, 1)
assert_check(pct_district == 100.0, f"100% account matchano un district ({pct_district}%)")

print("\n" + "=" * 80)
print("✅ ALL SILVER QUALITY CHECKS PASSED")
print("=" * 80)